In [1]:
from parameter.aparameter import AParameter
from strategy.strategy_factory import StrategyFactory
from transformer.transformer import Transformer
from backtester.backtester import Backtester
from processor.server_processor import ServerProcessor
from datetime import datetime, timedelta
from strategy.strategy import Strategy
import pandas as pd
from tqdm import tqdm
from database.adatabase import ADatabase
import matplotlib.pyplot as plt

In [2]:
db = ADatabase("sapling")

In [3]:
db.connect()
kpi = db.retrieve("kpi")
db.disconnect()

In [4]:
strategy_kpis = kpi.sort_values("return",ascending=False).groupby(["strategy"]).first().sort_values("return",ascending=False).reset_index()

In [5]:
strategy_kpis

,strategy,number_of_trades,standard_deviation,coefficient_of_variance,sharpe,return,holding_period,positions,stop_loss,ascending
0,KURTOSIS,103,62.8823,0.8363,2.8129,176.8811,5,1,1,False
1,STANDARD_DEVIATION,103,71.9416,2.5565,2.1411,154.0320,5,1,1,True
2,STOCHASTIC_OSCILLATOR,1030,54.2735,0.8111,2.2673,123.0518,5,10,1,False
3,RSI,515,64.2866,0.9041,1.7981,115.5939,5,5,1,True
4,BOLLINGER_WIDTH,103,48.7353,2.3741,2.3037,112.2698,5,1,1,True
5,COEFFICIENT_OF_VARIANCE,103,48.7353,2.3741,2.3037,112.2698,5,1,1,True
6,UNITY_AI,82,42.2233,0.7352,2.5793,108.9083,5,1,1,False
7,BOLLINGER,103,112.4775,0.8117,0.9548,107.3897,5,1,1,False
8,RANDOM,103,29.2913,0.6645,3.5445,103.8233,5,1,1,True
9,MACD,1030,38.4221,0.7248,2.4702,94.9111,5,10,1,True


In [6]:
db = ADatabase("sapling")
market = ADatabase("market")
market.connect()
tickers = market.retrieve("russell1000")["ticker"].values
market.disconnect()
strategy_kpis["tickers"] = [tickers for i in range(strategy_kpis.index.size)]
query = strategy_kpis.to_dict("records")[0]

In [7]:
analysis = []
db.connect()
recs = []
try:
    start = datetime.now() - timedelta(days=365.25)
    end = datetime.now() - timedelta(hours=24)
    parameter = AParameter()
    parameter.build(query)
    strategy = StrategyFactory.build(parameter)
    simulation = Transformer.local_transform(strategy,start,end)
    trades = Backtester.backtest(strategy,simulation)
    portfolio = Backtester.portfolio(trades)
    recommendations = Backtester.recommendations(trades)
    kpi = Backtester.kpi(trades,portfolio)
    results = ServerProcessor.server_format(strategy,trades,portfolio,recommendations,kpi)
    recs.extend(results["recommendations"])
except Exception as e:
    print(str(e))
db.disconnect()

In [8]:
recommendations = pd.DataFrame(recs)

In [9]:
recommendations

,date,buy_date,sell_date,ticker,adjclose,kurtosis
0,2024-02-09,2024-02-12,2024-02-16,SNA,262.43,8.0951


In [10]:
db.connect()
db.drop("recommendations")
db.store("recommendations",recommendations)
db.disconnect()